# 01 QC 预处理

面向 `6` 个胆囊癌 scRNA-seq 样本的严格质控 notebook。

## 本 notebook 做什么
- 按样本读取原始 `10x` 表达矩阵
- 计算核心 QC 指标
- 按样本分别应用 QC 阈值
- 检测并剔除 doublet
- 生成批次/样本效应诊断图，但不做整合校正
- 导出干净的 `qc_clean/*.h5ad` 对象和标准化结果表

## 本 notebook 不做什么
- 不做 Harmony / BBKNN / scVI 整合
- 不做细胞类型注释
- 不做 malignant inference / pseudo-bulk / CellChat / Monocle3
- 不输出生物学结论；这里只报告技术质控结果


## 1. 参数与输入

运行前请先完成：
1. 填写 `templates/sample_metadata_template.csv`
2. 复制并修改 `templates/sample_qc_thresholds_template.csv`
3. 将原始 `10x` 文件放到 `RAW_10X_ROOT`

推荐目录结构：

```text
project/
  data/raw/
    P01_T/
      matrix.mtx.gz
      barcodes.tsv.gz
      features.tsv.gz
    ...
  templates/
  notebooks/
```


In [17]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import sparse

try:
    import scrublet as scr
    SCRUBLET_AVAILABLE = True
except ImportError:
    scr = None
    SCRUBLET_AVAILABLE = False
    warnings.warn(
        "scrublet is not installed. QC and batch diagnostics can continue, but install scrublet before running the doublet step: pip install scrublet"
    )

sns.set_context("talk")
sns.set_style("whitegrid")
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=110, facecolor="white")

PROJECT_ROOT = Path("..").resolve()
RAW_10X_ROOT = PROJECT_ROOT / "data" / "raw"
METADATA_PATH = PROJECT_ROOT / "templates" / "sample_metadata_template.csv"
THRESHOLD_PATH = PROJECT_ROOT / "templates" / "sample_qc_thresholds_template.csv"
OUTPUT_ROOT = PROJECT_ROOT / "results"
FIGURE_DIR = OUTPUT_ROOT / "figures" / "qc"
TABLE_DIR = OUTPUT_ROOT / "tables" / "qc"
INTERMEDIATE_DIR = OUTPUT_ROOT / "intermediate" / "qc_clean"
MERGED_DIR = OUTPUT_ROOT / "intermediate"

MT_PREFIXES = ("MT-", "mt-")
EXPECTED_DOUBLET_RATE = 0.06
SCRUBLET_MIN_COUNTS = 2
SCRUBLET_MIN_CELLS = 3
SCRUBLET_MIN_GENE_VARIABILITY_PCTL = 85
SCRUBLET_N_PRIN_COMPS = 30
RANDOM_SEED = 20260309
LOW_CELL_COUNT_WARNING = 500


# ===== ???????? =====
def log_step(title: str) -> None:
    print(f"\n[{title}]")


def print_parameter_summary() -> None:
    parameter_summary = {
        "PROJECT_ROOT": str(PROJECT_ROOT),
        "RAW_10X_ROOT": str(RAW_10X_ROOT),
        "METADATA_PATH": str(METADATA_PATH),
        "THRESHOLD_PATH": str(THRESHOLD_PATH),
        "MT_PREFIXES": MT_PREFIXES,
        "EXPECTED_DOUBLET_RATE": EXPECTED_DOUBLET_RATE,
        "SCRUBLET_MIN_COUNTS": SCRUBLET_MIN_COUNTS,
        "SCRUBLET_MIN_CELLS": SCRUBLET_MIN_CELLS,
        "SCRUBLET_MIN_GENE_VARIABILITY_PCTL": SCRUBLET_MIN_GENE_VARIABILITY_PCTL,
        "SCRUBLET_N_PRIN_COMPS": SCRUBLET_N_PRIN_COMPS,
        "RANDOM_SEED": RANDOM_SEED,
        "LOW_CELL_COUNT_WARNING": LOW_CELL_COUNT_WARNING,
        "SCRUBLET_AVAILABLE": SCRUBLET_AVAILABLE,
    }
    print("Current parameter configuration:")
    for key, value in parameter_summary.items():
        print(f"- {key}: {value}")


for path in [FIGURE_DIR, TABLE_DIR, INTERMEDIATE_DIR, MERGED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

log_step("?? 1A????????")
print_parameter_summary()
print("Per-sample QC thresholds in use:")



[?? 1A????????]
Current parameter configuration:
- PROJECT_ROOT: D:\Codex\Project\PNI SC
- RAW_10X_ROOT: D:\Codex\Project\PNI SC\data\raw
- METADATA_PATH: D:\Codex\Project\PNI SC\templates\sample_metadata_template.csv
- THRESHOLD_PATH: D:\Codex\Project\PNI SC\templates\sample_qc_thresholds_template.csv
- MT_PREFIXES: ('MT-', 'mt-')
- EXPECTED_DOUBLET_RATE: 0.06
- SCRUBLET_MIN_COUNTS: 2
- SCRUBLET_MIN_CELLS: 3
- SCRUBLET_MIN_GENE_VARIABILITY_PCTL: 85
- SCRUBLET_N_PRIN_COMPS: 30
- RANDOM_SEED: 20260309
- LOW_CELL_COUNT_WARNING: 500
- SCRUBLET_AVAILABLE: True
Per-sample QC thresholds in use:


In [18]:
REQUIRED_METADATA_COLUMNS = [
    "sample_id",
    "patient_id",
    "pair_id",
    "tissue_type",
    "pni_level",
    "library_id",
    "sequencing_batch",
]

REQUIRED_THRESHOLD_COLUMNS = [
    "sample_id",
    "gene_low_cutoff",
    "gene_high_cutoff",
    "umi_low_cutoff",
    "umi_high_cutoff",
    "mt_cutoff",
    "expected_doublet_rate",
    "notes",
]


# ?? metadata / threshold ??????
def validate_columns(df: pd.DataFrame, required_columns: list[str], label: str) -> None:
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(f"{label} missing required columns: {missing}")


# ???? metadata
def load_metadata(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"metadata file not found: {path}")
    metadata = pd.read_csv(path)
    validate_columns(metadata, REQUIRED_METADATA_COLUMNS, "metadata")
    metadata = metadata.copy()
    metadata["sample_id"] = metadata["sample_id"].astype(str)
    return metadata


# ????? QC ??
def load_thresholds(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"threshold file not found: {path}")
    thresholds = pd.read_csv(path)
    validate_columns(thresholds, REQUIRED_THRESHOLD_COLUMNS, "thresholds")
    numeric_columns = [
        "gene_low_cutoff",
        "gene_high_cutoff",
        "umi_low_cutoff",
        "umi_high_cutoff",
        "mt_cutoff",
        "expected_doublet_rate",
    ]
    thresholds = thresholds.copy()
    thresholds["sample_id"] = thresholds["sample_id"].astype(str)
    for column in numeric_columns:
        thresholds[column] = pd.to_numeric(thresholds[column], errors="coerce")
    if thresholds[numeric_columns].isna().any().any():
        raise ValueError("threshold file contains non-numeric values; please check it.")
    return thresholds


# ? sample_id ???? 10x ??????????/?????
def find_10x_directory(sample_id: str, root: Path) -> Path:
    sample_aliases = [
        sample_id,
        sample_id.replace("_", "-"),
        sample_id.replace("-", "_"),
    ]
    sample_aliases = list(dict.fromkeys(sample_aliases))

    candidates = []
    for alias in sample_aliases:
        candidates.extend([
            root / alias,
            root / f"{alias}_filtered_feature_bc_matrix",
            root / alias / "filtered_feature_bc_matrix",
            root / alias / "outs" / "filtered_feature_bc_matrix",
            root / alias / "filtered_feature_bc_matrix.h5",
            root / f"{alias}.h5",
        ])

    existing = [candidate for candidate in candidates if candidate.exists()]
    if not existing:
        searched = ", ".join(str(candidate) for candidate in candidates)
        raise FileNotFoundError(
            f"sample_id={sample_id} has no readable 10x file or directory under {root}. Searched: {searched}"
        )
    return existing[0]


# ??????? 10x ????
def read_10x_sample(sample_id: str, root: Path) -> sc.AnnData:
    sample_path = find_10x_directory(sample_id, root)
    if sample_path.is_file() and sample_path.suffix == ".h5":
        adata = sc.read_10x_h5(sample_path)
    else:
        adata = sc.read_10x_mtx(sample_path, var_names="gene_symbols", make_unique=True)
    adata.var_names_make_unique()
    adata.obs_names = [f"{sample_id}:{barcode}" for barcode in adata.obs_names]
    adata.obs["barcode"] = adata.obs_names
    adata.uns["input_path"] = str(sample_path)
    return adata


# ????? metadata ?????
def add_sample_metadata(adata: sc.AnnData, row: pd.Series) -> sc.AnnData:
    for column, value in row.items():
        adata.obs[column] = value
    return adata


# ???? QC ??
def add_qc_metrics(adata: sc.AnnData, mt_prefixes: tuple[str, ...]) -> sc.AnnData:
    adata.var["mt"] = False
    for prefix in mt_prefixes:
        adata.var["mt"] = adata.var["mt"] | adata.var_names.str.startswith(prefix)
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
    adata.obs["percent_mt"] = adata.obs["pct_counts_mt"]
    return adata


# ????????? QC ??
def apply_sample_thresholds(adata: sc.AnnData, threshold_row: pd.Series) -> tuple[sc.AnnData, pd.DataFrame]:
    obs = adata.obs.copy()
    qc_pass = (
        (obs["n_genes_by_counts"] >= threshold_row["gene_low_cutoff"]) &
        (obs["n_genes_by_counts"] <= threshold_row["gene_high_cutoff"]) &
        (obs["total_counts"] >= threshold_row["umi_low_cutoff"]) &
        (obs["total_counts"] <= threshold_row["umi_high_cutoff"]) &
        (obs["percent_mt"] <= threshold_row["mt_cutoff"])
    )
    decision_df = obs[[
        "sample_id",
        "patient_id",
        "tissue_type",
        "pni_level",
        "n_genes_by_counts",
        "total_counts",
        "percent_mt",
    ]].copy()
    decision_df["passed_basic_qc"] = qc_pass.values
    filtered = adata[qc_pass].copy()
    filtered.obs["passed_basic_qc"] = True
    return filtered, decision_df


# ?? Scrublet ?????
def run_scrublet(adata: sc.AnnData, expected_doublet_rate: float) -> tuple[np.ndarray, np.ndarray]:
    if not SCRUBLET_AVAILABLE:
        raise ImportError("scrublet is not installed, so doublet detection cannot run. Please install it first: pip install scrublet")
    matrix = adata.X
    if sparse.issparse(matrix):
        counts_matrix = matrix.tocsc()
    else:
        counts_matrix = sparse.csc_matrix(matrix)
    scrub = scr.Scrublet(counts_matrix, expected_doublet_rate=expected_doublet_rate)
    scores, predicted_doublets = scrub.scrub_doublets(
        min_counts=SCRUBLET_MIN_COUNTS,
        min_cells=SCRUBLET_MIN_CELLS,
        min_gene_variability_pctl=SCRUBLET_MIN_GENE_VARIABILITY_PCTL,
        n_prin_comps=SCRUBLET_N_PRIN_COMPS,
        use_approx_neighbors=False,
    )
    return scores, predicted_doublets


def save_qc_panel(adata: sc.AnnData, sample_id: str, outdir: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    sns.histplot(adata.obs["n_genes_by_counts"], bins=50, ax=axes[0, 0])
    axes[0, 0].set_title(f"{sample_id} n_genes_by_counts")
    sns.histplot(adata.obs["total_counts"], bins=50, ax=axes[0, 1])
    axes[0, 1].set_title(f"{sample_id} total_counts")
    sns.histplot(adata.obs["percent_mt"], bins=50, ax=axes[1, 0])
    axes[1, 0].set_title(f"{sample_id} percent_mt")
    sns.scatterplot(
        data=adata.obs,
        x="total_counts",
        y="n_genes_by_counts",
        hue="percent_mt",
        palette="viridis",
        linewidth=0,
        s=10,
        ax=axes[1, 1],
    )
    axes[1, 1].set_title(f"{sample_id} counts vs genes")
    plt.tight_layout()
    fig.savefig(outdir / f"{sample_id}_qc_panel.png", bbox_inches="tight")
    plt.close(fig)


def save_doublet_panel(adata: sc.AnnData, sample_id: str, outdir: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.histplot(adata.obs["doublet_score"], bins=50, ax=axes[0])
    axes[0].set_title(f"{sample_id} doublet_score")
    sns.scatterplot(
        data=adata.obs,
        x="total_counts",
        y="n_genes_by_counts",
        hue="predicted_doublet",
        palette={False: "#4daf4a", True: "#e41a1c"},
        linewidth=0,
        s=12,
        ax=axes[1],
    )
    axes[1].set_title(f"{sample_id} predicted doublets")
    plt.tight_layout()
    fig.savefig(outdir / f"{sample_id}_doublet_panel.png", bbox_inches="tight")
    plt.close(fig)


# ??????? QC ?????????
def summarise_sample(
    raw_adata: sc.AnnData,
    qc_adata: sc.AnnData,
    clean_adata: sc.AnnData,
    threshold_row: pd.Series,
) -> dict:
    predicted_rate = float(qc_adata.obs["predicted_doublet"].mean()) if qc_adata.n_obs else np.nan
    return {
        "sample_id": threshold_row["sample_id"],
        "patient_id": threshold_row["patient_id"],
        "tissue_type": threshold_row["tissue_type"],
        "pni_level": threshold_row["pni_level"],
        "n_cells_raw": int(raw_adata.n_obs),
        "n_cells_after_qc": int(qc_adata.n_obs),
        "n_cells_after_doublet_removal": int(clean_adata.n_obs),
        "gene_low_cutoff": float(threshold_row["gene_low_cutoff"]),
        "gene_high_cutoff": float(threshold_row["gene_high_cutoff"]),
        "umi_low_cutoff": float(threshold_row["umi_low_cutoff"]),
        "umi_high_cutoff": float(threshold_row["umi_high_cutoff"]),
        "mt_cutoff": float(threshold_row["mt_cutoff"]),
        "doublet_rate_predicted": predicted_rate,
        "risk_low_cell_count": bool(clean_adata.n_obs < LOW_CELL_COUNT_WARNING),
    }


# ???/????????????
def prepare_for_batch_diagnostics(adata: sc.AnnData) -> sc.AnnData:
    diag = adata.copy()
    sc.pp.normalize_total(diag, target_sum=1e4)
    sc.pp.log1p(diag)
    sc.pp.highly_variable_genes(diag, n_top_genes=3000, subset=True, flavor="seurat")
    sc.pp.scale(diag, max_value=10)
    sc.tl.pca(diag, svd_solver="arpack", random_state=RANDOM_SEED)
    sc.pp.neighbors(diag, n_neighbors=15, n_pcs=30, random_state=RANDOM_SEED)
    sc.tl.umap(diag, random_state=RANDOM_SEED)
    return diag

In [19]:
log_step("Step 1B: load metadata and QC thresholds")
metadata = load_metadata(METADATA_PATH)
thresholds = load_thresholds(THRESHOLD_PATH)

metadata = metadata.merge(thresholds, on="sample_id", how="left", validate="one_to_one")
if metadata[REQUIRED_THRESHOLD_COLUMNS[1:]].isna().any().any():
    missing_samples = metadata.loc[
        metadata[REQUIRED_THRESHOLD_COLUMNS[1:]].isna().any(axis=1), "sample_id"
    ].tolist()
    raise ValueError(f"missing QC thresholds for samples: {missing_samples}")

print(f"Completed: loaded metadata for {metadata.shape[0]} samples.")
print("Sample list:", ", ".join(metadata["sample_id"].tolist()))
print("Per-sample QC thresholds in use:")
print(thresholds[["sample_id", "gene_low_cutoff", "gene_high_cutoff", "umi_low_cutoff", "umi_high_cutoff", "mt_cutoff", "expected_doublet_rate"]])

metadata



[Step 1B: load metadata and QC thresholds]
Completed: loaded metadata for 6 samples.
Sample list: P01_T, P01_A, P02_T, P02_A, P03_T, P03_A
Per-sample QC thresholds in use:
  sample_id  gene_low_cutoff  gene_high_cutoff  umi_low_cutoff  \
0     P01_T              300              7000             500   
1     P01_A              300              8000             500   
2     P02_T              300              8000             500   
3     P02_A              300              9000             500   
4     P03_T              300              6500             500   
5     P03_A              300              7000             500   

   umi_high_cutoff  mt_cutoff  expected_doublet_rate  
0            40000         20                   0.06  
1            50000         24                   0.05  
2            50000         24                   0.06  
3            75000         24                   0.05  
4            40000         20                   0.06  
5            40000         24     

,sample_id,patient_id,pair_id,tissue_type,pni_level,library_id,sequencing_batch,sex,age,pathology_note,gene_low_cutoff,gene_high_cutoff,umi_low_cutoff,umi_high_cutoff,mt_cutoff,expected_doublet_rate,notes
0,P01_T,P01,P01,tumor,high,LIB001,B1,NaN,NaN,,300,7000,500,40000,20,0.06,tumor sample threshold placeholder
1,P01_A,P01,P01,adjacent,high,LIB002,B1,NaN,NaN,,300,8000,500,50000,24,0.05,adjacent sample threshold placeholder
2,P02_T,P02,P02,tumor,medium,LIB003,B1,NaN,NaN,,300,8000,500,50000,24,0.06,tumor sample threshold placeholder
3,P02_A,P02,P02,adjacent,medium,LIB004,B1,NaN,NaN,,300,9000,500,75000,24,0.05,adjacent sample threshold placeholder
4,P03_T,P03,P03,tumor,low,LIB005,B1,NaN,NaN,,300,6500,500,40000,20,0.06,tumor sample threshold placeholder
5,P03_A,P03,P03,adjacent,low,LIB006,B1,NaN,NaN,,300,7000,500,40000,24,0.05,adjacent sample threshold placeholder


## 2. 原始输入与 QC 指标

本部分逐样本读取数据并计算：
- `n_genes_by_counts`
- `total_counts`
- `percent_mt`

同时保存每个样本的 QC 面板图，供后续人工复核阈值是否合理。


In [20]:
log_step("Step 2: load raw matrices and compute QC metrics")
raw_adatas: dict[str, sc.AnnData] = {}
qc_previews: list[dict] = []

for row in metadata.to_dict(orient="records"):
    sample_id = row["sample_id"]
    print(f"[LOAD] {sample_id}")
    print(f"  - input path alias search root: {RAW_10X_ROOT}")
    adata = read_10x_sample(sample_id=sample_id, root=RAW_10X_ROOT)
    adata = add_sample_metadata(adata, pd.Series(row))
    adata = add_qc_metrics(adata, MT_PREFIXES)
    raw_adatas[sample_id] = adata
    save_qc_panel(adata, sample_id=sample_id, outdir=FIGURE_DIR)
    qc_previews.append({
        "sample_id": sample_id,
        "n_cells_raw": adata.n_obs,
        "n_genes_detected": adata.n_vars,
        "median_genes": float(np.median(adata.obs["n_genes_by_counts"])),
        "median_umis": float(np.median(adata.obs["total_counts"])),
        "median_percent_mt": float(np.median(adata.obs["percent_mt"])),
        "input_path": adata.uns["input_path"],
    })

qc_preview_df = pd.DataFrame(qc_previews).sort_values("sample_id")
qc_preview_df.to_csv(TABLE_DIR / "sample_qc_preview.tsv", sep="\t", index=False)
qc_preview_df
print(f"Completed: loaded {len(raw_adatas)} samples and exported sample_qc_preview.tsv plus per-sample QC panels.")



[Step 2: load raw matrices and compute QC metrics]
[LOAD] P01_T
  - input path alias search root: D:\Codex\Project\PNI SC\data\raw
[LOAD] P01_A
  - input path alias search root: D:\Codex\Project\PNI SC\data\raw
[LOAD] P02_T
  - input path alias search root: D:\Codex\Project\PNI SC\data\raw
[LOAD] P02_A
  - input path alias search root: D:\Codex\Project\PNI SC\data\raw
[LOAD] P03_T
  - input path alias search root: D:\Codex\Project\PNI SC\data\raw
[LOAD] P03_A
  - input path alias search root: D:\Codex\Project\PNI SC\data\raw
Completed: loaded 6 samples and exported sample_qc_preview.tsv plus per-sample QC panels.


## 3. 按样本阈值过滤

过滤使用每个样本独立阈值，只针对技术质量问题：
- 检测到的基因数过低或过高
- UMI 数过低或过高
- 线粒体比例过高

不会因为怀疑存在稀有群而放宽阈值。


In [21]:
log_step("Step 3: apply sample-specific basic QC filters")
qc_filtered_adatas: dict[str, sc.AnnData] = {}
qc_decision_tables: list[pd.DataFrame] = []
threshold_records: list[dict] = []

for row in metadata.to_dict(orient="records"):
    sample_id = row["sample_id"]
    adata = raw_adatas[sample_id]
    qc_filtered, decisions = apply_sample_thresholds(adata, pd.Series(row))
    qc_filtered_adatas[sample_id] = qc_filtered
    decisions["filter_stage"] = "basic_qc"
    qc_decision_tables.append(decisions)
    threshold_records.append({
        "sample_id": sample_id,
        "patient_id": row["patient_id"],
        "tissue_type": row["tissue_type"],
        "pni_level": row["pni_level"],
        "gene_low_cutoff": row["gene_low_cutoff"],
        "gene_high_cutoff": row["gene_high_cutoff"],
        "umi_low_cutoff": row["umi_low_cutoff"],
        "umi_high_cutoff": row["umi_high_cutoff"],
        "mt_cutoff": row["mt_cutoff"],
        "expected_doublet_rate": row["expected_doublet_rate"],
        "notes": row["notes"],
        "n_cells_raw": adata.n_obs,
        "n_cells_after_qc": qc_filtered.n_obs,
        "retention_after_qc": round(qc_filtered.n_obs / adata.n_obs, 4) if adata.n_obs else np.nan,
    })
    print(f"[QC] {sample_id}: {adata.n_obs} -> {qc_filtered.n_obs}")

threshold_df = pd.DataFrame(threshold_records).sort_values("sample_id")
threshold_df.to_csv(TABLE_DIR / "sample_thresholds_applied.tsv", sep="\t", index=False)

basic_qc_decisions_df = pd.concat(qc_decision_tables, axis=0)
basic_qc_decisions_df.to_csv(TABLE_DIR / "cell_level_basic_qc_decisions.tsv", sep="\t", index=True)

threshold_df
print("Completed: basic QC filtering finished. Sample-level summary:")
print(threshold_df[["sample_id", "n_cells_raw", "n_cells_after_qc", "retention_after_qc", "mt_cutoff", "expected_doublet_rate"]])



[Step 3: apply sample-specific basic QC filters]
[QC] P01_T: 9619 -> 8253
[QC] P01_A: 8299 -> 7897
[QC] P02_T: 6867 -> 5612
[QC] P02_A: 6304 -> 5358
[QC] P03_T: 18395 -> 16835
[QC] P03_A: 8348 -> 7577
Completed: basic QC filtering finished. Sample-level summary:
  sample_id  n_cells_raw  n_cells_after_qc  retention_after_qc  mt_cutoff  \
1     P01_A         8299              7897              0.9516         24   
0     P01_T         9619              8253              0.8580         20   
3     P02_A         6304              5358              0.8499         24   
2     P02_T         6867              5612              0.8172         24   
5     P03_A         8348              7577              0.9076         24   
4     P03_T        18395             16835              0.9152         20   

   expected_doublet_rate  
1                   0.05  
0                   0.06  
3                   0.05  
2                   0.06  
5                   0.05  
4                   0.06  


## 4. Doublet 检测与剔除

本部分对每个样本运行 `Scrublet`：
- 记录 doublet score
- 记录预测标签
- 直接剔除预测为 doublet 的细胞

如果剩余细胞数过少，请优先回看 QC 阈值和原始数据质量，而不是保留预测 doublet。


In [ ]:
log_step("Step 4: run Scrublet and remove predicted doublets")
clean_adatas: dict[str, sc.AnnData] = {}
doublet_tables: list[pd.DataFrame] = []
summary_records: list[dict] = []

for row in metadata.to_dict(orient="records"):
    sample_id = row["sample_id"]
    qc_adata = qc_filtered_adatas[sample_id].copy()
    expected_rate = float(row["expected_doublet_rate"])
    print(f"[DOUBLET] {sample_id} expected_rate={expected_rate}")

    scores, predicted = run_scrublet(qc_adata, expected_doublet_rate=expected_rate)
    qc_adata.obs["doublet_score"] = scores
    #qc_adata.obs["predicted_doublet"] = predicted.astype(bool)
    manual_cutoff_map = {
    "P01_A": 0.25,
    "P01_T": 0.25,
    "P02_A": 0.30,
    "P02_T": 0.30,
    "P03_A": 0.25,
    "P03_T": 0.30,
    }
    cutoff = manual_cutoff_map[sample_id]
    qc_adata.obs["predicted_doublet"] = qc_adata.obs["doublet_score"] >= cutoff

    save_doublet_panel(qc_adata, sample_id=sample_id, outdir=FIGURE_DIR)

    doublet_table = qc_adata.obs[[
        "sample_id",
        "patient_id",
        "tissue_type",
        "pni_level",
        "n_genes_by_counts",
        "total_counts",
        "percent_mt",
        "doublet_score",
        "predicted_doublet",
    ]].copy()
    doublet_table["cell_barcode"] = qc_adata.obs_names
    doublet_tables.append(doublet_table)

    clean_adata = qc_adata[~qc_adata.obs["predicted_doublet"]].copy()
    clean_adata.obs["is_doublet"] = False
    clean_adata.obs["passed_basic_qc"] = True
    clean_adata.obs["kept_after_qc_pipeline"] = True
    clean_adatas[sample_id] = clean_adata

    summary_records.append(summarise_sample(
        raw_adata=raw_adatas[sample_id],
        qc_adata=qc_adata,
        clean_adata=clean_adata,
        threshold_row=pd.Series(row),
    ))
    print(f"[KEEP] {sample_id}: {qc_adata.n_obs} -> {clean_adata.n_obs}")

sample_qc_summary = pd.DataFrame(summary_records).sort_values("sample_id")
sample_qc_summary.to_csv(TABLE_DIR / "sample_qc_summary.tsv", sep="\t", index=False)

doublet_calls = pd.concat(doublet_tables, axis=0)
doublet_calls.to_csv(TABLE_DIR / "doublet_calls.tsv", sep="\t", index=False)

sample_qc_summary
print("Completed: doublet detection and removal finished. Sample-level summary:")
print(sample_qc_summary[["sample_id", "n_cells_after_qc", "n_cells_after_doublet_removal", "doublet_rate_predicted", "risk_low_cell_count"]])



[Step 4: run Scrublet and remove predicted doublets]
[DOUBLET] P01_T expected_rate=0.06
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.64
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 4.2%
Overall doublet rate:
	Expected   = 6.0%
	Estimated  = 1.2%
Elapsed time: 3.5 seconds
[KEEP] P01_T: 8253 -> 8118
[DOUBLET] P01_A expected_rate=0.05
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.59
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 5.7%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 1.3%
Elapsed time: 3.7 seconds
[KEEP] P01_A: 7897 -> 7769
[DOUBLET] P02_T expected_rate=0.06
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet 

## 5. 批次效应诊断

本部分将 QC 后的样本拼接，仅用于诊断：
- `sample_id` 是否主导嵌入结构
- `patient_id` 与 `tissue_type` 是否仍可见
- `sequencing_batch` 是否解释主要变异
- 技术指标是否与样本来源强耦合

这里的降维结果仅用于诊断，不代表正式整合后的结果。


In [23]:
log_step("Step 5: generate batch-effect diagnostics")
print("Batch diagnostic parameters: n_top_genes=3000, flavor=seurat, n_neighbors=15, n_pcs=30")
ordered_samples = metadata["sample_id"].tolist()
merged = sc.concat(
    [clean_adatas[sample_id] for sample_id in ordered_samples],
    join="outer",
    label="source_sample",
    keys=ordered_samples,
    index_unique=None,
)
merged.write_h5ad(MERGED_DIR / "qc_merged_unintegrated.h5ad")

batch_diag = prepare_for_batch_diagnostics(merged)
batch_diag.write_h5ad(MERGED_DIR / "qc_merged_unintegrated_diagnostic.h5ad")

color_fields = [
    "sample_id",
    "patient_id",
    "tissue_type",
    "sequencing_batch",
    "n_genes_by_counts",
    "total_counts",
    "percent_mt",
]

for field in color_fields:
    sc.pl.umap(
        batch_diag,
        color=field,
        show=False,
        frameon=False,
        save=False,
    )
    plt.gcf().savefig(FIGURE_DIR / f"batch_diagnostic_umap__{field}.png", bbox_inches="tight")
    plt.close(plt.gcf())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(data=batch_diag.obs, x="sample_id", y="n_genes_by_counts", ax=axes[0])
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_title("Genes by sample")
sns.boxplot(data=batch_diag.obs, x="sample_id", y="total_counts", ax=axes[1])
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_title("UMIs by sample")
sns.boxplot(data=batch_diag.obs, x="sample_id", y="percent_mt", ax=axes[2])
axes[2].tick_params(axis="x", rotation=45)
axes[2].set_title("Percent MT by sample")
plt.tight_layout()
fig.savefig(FIGURE_DIR / "technical_metrics_by_sample.png", bbox_inches="tight")
plt.close(fig)

risk_samples = sample_qc_summary.loc[
    sample_qc_summary["risk_low_cell_count"], "sample_id"
].tolist()
if risk_samples:
    warnings.warn(f"low retained cell count after QC for samples: {risk_samples}")
else:
    print("No samples fell below the low-cell-count warning threshold.")
print(f"Completed: batch diagnostic object contains {batch_diag.n_obs} cells and {batch_diag.n_vars} genes.")
print("Exported: UMAP diagnostic plots and technical_metrics_by_sample.png")



[Step 5: generate batch-effect diagnostics]
Batch diagnostic parameters: n_top_genes=3000, flavor=seurat, n_neighbors=15, n_pcs=30
normalizing counts per cell
    finished (0:00:00)
extracting highly variable genes
    finished (0:00:00)


d:\Codex\Project\PNI SC\.conda\Lib\functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


computing PCA
    with n_comps=50
    finished (0:00:07)
computing neighbors
    using 'X_pca' with n_pcs = 30
    finished (0:00:02)
computing UMAP
    finished (0:00:21)
No samples fell below the low-cell-count warning threshold.
Completed: batch diagnostic object contains 50844 cells and 3000 genes.
Exported: UMAP diagnostic plots and technical_metrics_by_sample.png


## 6. 结果导出

本 notebook 导出：
- `results/intermediate/qc_clean/<sample>.h5ad`
- `results/tables/qc/sample_qc_summary.tsv`
- `results/tables/qc/sample_thresholds_applied.tsv`
- `results/tables/qc/doublet_calls.tsv`
- `results/tables/qc/qc_clean_object_index.tsv`
- `results/figures/qc/*.png`


In [24]:
log_step("Step 6: export cleaned objects and result tables")
object_index_records = []

for sample_id, adata in clean_adatas.items():
    output_path = INTERMEDIATE_DIR / f"{sample_id}.h5ad"
    adata.write_h5ad(output_path)
    summary_row = sample_qc_summary.loc[sample_qc_summary["sample_id"] == sample_id].iloc[0]
    object_index_records.append({
        "sample_id": sample_id,
        "patient_id": summary_row["patient_id"],
        "tissue_type": summary_row["tissue_type"],
        "pni_level": summary_row["pni_level"],
        "object_path": str(output_path),
        "n_cells": int(adata.n_obs),
    })

object_index_df = pd.DataFrame(object_index_records).sort_values("sample_id")
object_index_df.to_csv(TABLE_DIR / "qc_clean_object_index.tsv", sep="\t", index=False)

summary_payload = {
    "raw_10x_root": str(RAW_10X_ROOT),
    "metadata_path": str(METADATA_PATH),
    "threshold_path": str(THRESHOLD_PATH),
    "expected_default_doublet_rate": EXPECTED_DOUBLET_RATE,
    "random_seed": RANDOM_SEED,
    "outputs": {
        "figure_dir": str(FIGURE_DIR),
        "table_dir": str(TABLE_DIR),
        "intermediate_dir": str(INTERMEDIATE_DIR),
    },
}
with open(TABLE_DIR / "run_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(summary_payload, handle, ensure_ascii=False, indent=2)

print("Export complete.")
object_index_df


[Step 6: export cleaned objects and result tables]
Export complete.


,sample_id,patient_id,tissue_type,pni_level,object_path,n_cells
1,P01_A,P01,adjacent,high,D:\Codex\Project\PNI SC\results\intermediate\q...,7769
0,P01_T,P01,tumor,high,D:\Codex\Project\PNI SC\results\intermediate\q...,8118
3,P02_A,P02,adjacent,medium,D:\Codex\Project\PNI SC\results\intermediate\q...,5293
2,P02_T,P02,tumor,medium,D:\Codex\Project\PNI SC\results\intermediate\q...,5556
5,P03_A,P03,adjacent,low,D:\Codex\Project\PNI SC\results\intermediate\q...,7469
4,P03_T,P03,tumor,low,D:\Codex\Project\PNI SC\results\intermediate\q...,16639


## 人工复核清单

- 查看 `sample_qc_preview.tsv` 和每个 `*_qc_panel.png`，确认单样本阈值是否合理
- 查看 `sample_qc_summary.tsv`，确认保留率和预测 doublet 比例
- 查看 `batch_diagnostic_umap__*.png`，判断样本效应是否强于 `tissue_type`
- 如果某个样本被标记为低细胞数风险，请在进入 `02_integration_annotation` 前先复核阈值与原始文库质量
